In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!git clone -b feature/phase-5-training-evaluation https://github.com/SanduniHerath/diabetic-retinopathy-cv-assignment.git
%cd diabetic-retinopathy-cv-assignment

!pip install torch torchvision numpy matplotlib opencv-python scikit-learn datasets --quiet

Cloning into 'diabetic-retinopathy-cv-assignment'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 290 (delta 42), reused 73 (delta 27), pack-reused 190 (from 1)
Receiving objects: 100% (290/290), 45.05 MiB | 31.99 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/kaggle/working/diabetic-retinopathy-cv-assignment


In [2]:
!ls /kaggle/input/

competitions


In [4]:
!ls /kaggle/input/competitions/

aptos2019-blindness-detection


In [9]:
# Create only the missing subfolder
!mkdir -p data/raw/train_images

In [15]:
!rm -rf data/raw/train_images

In [35]:
!python src/organize_dataset.py \
  --csv /kaggle/input/competitions/aptos2019-blindness-detection/train.csv \
  --images /kaggle/input/competitions/aptos2019-blindness-detection/train_images \
  --out data/organized


[1] Loading CSV from: /kaggle/input/competitions/aptos2019-blindness-detection/train.csv
     Loaded 3662 records | Columns: ['id_code', 'diagnosis']

[2] Creating class directories under: data/organized
     No_DR                -> data/organized/No_DR
     Mild                 -> data/organized/Mild
     Moderate             -> data/organized/Moderate
     Severe               -> data/organized/Severe
     Proliferative_DR     -> data/organized/Proliferative_DR

[3] Organising 3662 images into class folders ...
  Organised 500/3662 images ...
  Organised 1000/3662 images ...
  Organised 1500/3662 images ...
  Organised 2000/3662 images ...
  Organised 2500/3662 images ...
  Organised 3000/3662 images ...
  Organised 3500/3662 images ...

  CLASS DISTRIBUTION – APTOS 2019 Retinopathy Dataset
  Label  Class Name             Count      Pct
-------------------------------------------------------
  0     No_DR                   1805    49.3%  ########################
  1     Mild        

In [36]:
!python src/split_dataset.py

PHASE 1: STRATIFIED TRAIN / VAL / TEST DATASET SPLIT
  Input Directory   : data/organized
  Output Directory  : data/split
  Ratios (T/V/Te)   : 0.70 / 0.15 / 0.15
  Random Seed       : 42
  Dry Run           : False
--------------------------------------------------------------------------------
[INFO] Discovered 3662 total images across 5 classes.

           APTOS 2019 DATASET STRATIFIED TRAIN / VAL / TEST SPLIT SUMMARY
Label  | Class Name        | Total   | Train (70%)    | Val (15%)      | Test (15%)    
-------------------------------------------------------------------------------------
0      | No_DR             | 1805    |  1264 ( 70.0%) |   271 ( 15.0%) |   270 ( 15.0%)
1      | Mild              | 370     |   259 ( 70.0%) |    56 ( 15.1%) |    55 ( 14.9%)
2      | Moderate          | 999     |   699 ( 70.0%) |   150 ( 15.0%) |   150 ( 15.0%)
3      | Severe            | 193     |   135 ( 69.9%) |    29 ( 15.0%) |    29 ( 15.0%)
4      | Proliferative_DR  | 295     |   206 ( 

In [37]:
!rm -rf data/organized 

In [38]:
!python src/add_external_data.py --max-per-class 500


  EYEPACS SUPPLEMENTARY DATA INGESTION (PHASE 5 EXTENSION)

[1] Verifying dataset split integrity under: /kaggle/working/diabetic-retinopathy-cv-assignment/data/split
  [GUARD] Validation (/kaggle/working/diabetic-retinopathy-cv-assignment/data/split/val) and Test (/kaggle/working/diabetic-retinopathy-cv-assignment/data/split/test) splits are strictly locked.
     Target Directory : /kaggle/working/diabetic-retinopathy-cv-assignment/data/split/train
     Isolation Policy : Val and Test directories are NEVER modified.

[2] Initial training counts baseline:
     Class 0 (No_DR           ):  1264 images
     Class 1 (Mild            ):   259 images
     Class 2 (Moderate        ):   699 images
     Class 3 (Severe          ):   135 images
     Class 4 (Proliferative_DR):   206 images

[3] Ingesting EyePACS supplementary samples ...

[Engine] Initializing Hugging Face dataset: Lelihu/eyepacs
         Target Labels : [1, 3, 4] -> ['Mild', 'Severe', 'Proliferative_DR']
         Quota Limit 

In [41]:
!mkdir -p /kaggle/working/hp_results

!python src/train.py \
  --data-root data/split \
  --output-dir /kaggle/working/hp_results \
  --backbone efficientnet_b0 \
  --batch-size 32 \
  --num-workers 2 \
  --fast-loader \
  --hp-search


[Device] Using: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB

[Mode] Running hyperparameter grid search ...

############################################################
  HP Search Run 1/4: run01_s1lr1e-03_s2lr1e-04_ub3
############################################################
[Data] Fast DataLoader active: using native PyTorch transforms (bypassing CPU HoughCircles).

 [TRAINING SET] CLINICAL CLASS-TO-INDEX VERIFICATION
  Index  Class Name         Grade    Status      
  --------------------------------------------------
  0      No_DR              Grade 0  MATCH (OK)  
  1      Mild               Grade 1  MATCH (OK)  
  2      Moderate           Grade 2  MATCH (OK)  
  3      Severe             Grade 3  MATCH (OK)  
  4      Proliferative_DR   Grade 4  MATCH (OK)  
  Sample counts by clinical class in Training Set:
    Class 0 (No_DR             ):  1264 samples ( 31.1%)
    Class 1 (Mild              ):   759 samples ( 18.7%)
    Class 2 (Moderate          ):   699 samples ( 17.2%)
   

In [43]:
import shutil
from IPython.display import FileLink

# 1. Zip the entire hp_results folder (CSV + all 4 run folders)
shutil.make_archive('/kaggle/working/hp_results', 'zip', '/kaggle/working/hp_results')

# 2. Generate a clickable direct download link in the notebook
FileLink(r'hp_results.zip')

/kaggle/working/diabetic-retinopathy-cv-assignment/hp_results.zip

In [ ]:
/kaggle/working/hp_results

In [34]:
!rm -rf data/split

In [29]:
!rm -rf reports/dataset_overview